# Lab 04 - Bronze Layer: Raw Data Ingestion (SOLUTION)

## Building the Foundation of Your Lakehouse

---

### Learning Objectives

By the end of this lab, you will be able to:

1. **Understand** the Bronze Layer role in the Medallion Architecture
2. **Configure** ADLS Gen2 connectivity in Azure Databricks
3. **Implement** batch ingestion with audit columns
4. **Configure** Auto Loader for incremental file processing


---

### Medallion Architecture Overview

In [1]:
displayHTML("""
<div style="background: #1B3A4B; border-radius: 12px; padding: 30px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #FFFFFF; letter-spacing: 2px;">MEDALLION ARCHITECTURE</h3>
  <div style="display: flex; justify-content: center; align-items: center; gap: 8px; flex-wrap: wrap;">
    <!-- Source -->
    <div style="text-align: center;">
      <div style="background: #FFFFFF; border: 2px solid #E8E5E0; border-radius: 10px; padding: 16px 20px; min-width: 110px;">
        <div style="width: 36px; height: 36px; background: #E8E5E0; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #1B3A4B; font-size: 16px;">S</div>
        <div style="font-weight: bold; font-size: 13px; color: #1B1F24;">Source</div>
        <div style="font-size: 11px; color: #757575; margin-top: 4px;">ADLS Gen2<br/>Raw JSON</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #CD7F32;">&#8594;</div>
    <!-- Bronze -->
    <div style="text-align: center;">
      <div style="background: #FFF3E0; border: 3px solid #CD7F32; border-radius: 10px; padding: 16px 20px; min-width: 110px; box-shadow: 0 0 15px rgba(205,127,50,0.3);">
        <div style="width: 36px; height: 36px; background: #CD7F32; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #FFFFFF; font-size: 16px;">B</div>
        <div style="font-weight: bold; font-size: 14px; color: #CD7F32;">BRONZE</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">Raw + Audit<br/>Append-Only</div>
        <div style="font-size: 10px; background: #FF3621; color: #FFFFFF; border-radius: 4px; padding: 2px 8px; margin-top: 6px; font-weight: bold; display: inline-block;">YOU ARE HERE</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #757575;">&#8594;</div>
    <!-- Silver -->
    <div style="text-align: center;">
      <div style="background: #F5F5F5; border: 2px solid #9E9E9E; border-radius: 10px; padding: 16px 20px; min-width: 110px;">
        <div style="width: 36px; height: 36px; background: #9E9E9E; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #FFFFFF; font-size: 16px;">S</div>
        <div style="font-weight: bold; font-size: 13px; color: #757575;">SILVER</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">Cleaned<br/>Validated</div>
      </div>
    </div>
    <div style="font-size: 22px; color: #757575;">&#8594;</div>
    <!-- Gold -->
    <div style="text-align: center;">
      <div style="background: #FFFDE7; border: 2px solid #F9A825; border-radius: 10px; padding: 16px 20px; min-width: 110px;">
        <div style="width: 36px; height: 36px; background: #F9A825; border-radius: 50%; margin: 0 auto 8px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: #1B1F24; font-size: 16px;">G</div>
        <div style="font-weight: bold; font-size: 13px; color: #F9A825;">GOLD</div>
        <div style="font-size: 11px; color: #1B1F24; margin-top: 4px;">Aggregated<br/>Business-Ready</div>
      </div>
    </div>
  </div>
</div>
""")

MEDALLION ARCHITECTURE 
 
 <!-- Source -->
 
 
 S 
 Source 
 ADLS Gen2 Raw JSON 
 
 
 → 
 <!-- Bronze -->
 
 
 B 
 BRONZE 
 Raw + Audit Append-Only 
 YOU ARE HERE 
 
 
 → 
 <!-- Silver -->
 
 
 S 
 SILVER 
 Cleaned Validated 
 
 
 → 
 <!-- Gold -->
 
 
 G 
 GOLD 
 Aggregated Business-Ready

---

## Section 1: Environment Configuration

This section configures your connection to ADLS Gen2 and defines all paths used throughout the lab. **You must update the placeholder values** with the credentials provided by your instructor.

In [ ]:
# ============================================================
# CONFIGURATION (UC External Location handles ADLS auth)
# ============================================================

STORAGE_ACCOUNT = ""  # <-- Enter your ADLS storage account name
CONTAINER = "data"  # <-- Enter your container name
WS_KEY = ""  # Leave empty (used for multi-user isolation only)

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net" + (f"/{WS_KEY}" if WS_KEY else "")
SOURCE_PATH = f"{BASE_PATH}/raw/placements"
SOURCE_INCREMENTAL = f"{BASE_PATH}/raw/placements_incremental"
BRONZE_PATH = f"{BASE_PATH}/bronze/placements"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_placements"
REFERENCE_PATH = f"{BASE_PATH}/raw/entities"
DATABASE_NAME = "bronze"

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Workspace Key: {WS_KEY}")
print(f"Base Path:     {BASE_PATH}")
print(f"Source Path:   {SOURCE_PATH}")
print(f"Bronze Path:   {BRONZE_PATH}")

In [ ]:
# ============================================================
# ADLS Gen2 ACCESS VERIFICATION
# ============================================================

try:
    files = dbutils.fs.ls(SOURCE_PATH)
    print(f"Access verified! Found {len(files)} items in source path")
    for f in files[:5]:
        print(f"  - {f.name} ({f.size} bytes)")
except Exception as e:
    print(f"Access error: {e}")
    print("\nEnsure setup notebook 00 was run first (creates External Location).")

In [ ]:
# Create database
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
spark.sql(f"USE {DATABASE_NAME}")
print(f"Database '{DATABASE_NAME}' ready")

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">Key Takeaway:</strong> <span style="color: #1B1F24;">A well-structured configuration cell with clear variable names and access verification makes your notebook portable across environments and easy to debug when things go wrong.</span>
</div>
""")

Key Takeaway: A well-structured configuration cell with clear variable names and access verification makes your notebook portable across environments and easy to debug when things go wrong.

---

## Section 2: Understanding the Bronze Layer

> *"The Bronze layer is your insurance policy. By keeping data exactly as it arrived from the source, you can always trace back to the original if anything goes wrong downstream."*

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Data Flow: Source to Bronze</h3>
  <div style="display: flex; justify-content: center; align-items: center; gap: 0; flex-wrap: wrap;">
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 14px 20px; min-width: 110px;">
        <div style="font-weight: bold; color: #1B1F24;">Source</div>
        <div style="font-size: 11px; color: #757575;">JSON in ADLS</div>
      </div>
    </div>
    <div style="font-size: 24px; color: #CD7F32; padding: 0 8px;">&#10132;</div>
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFF3E0; border: 3px solid #CD7F32; border-radius: 8px; padding: 14px 20px; min-width: 110px; box-shadow: 0 4px 12px rgba(205,127,50,0.3);">
        <div style="font-weight: bold; font-size: 15px; color: #CD7F32;">BRONZE</div>
        <div style="font-size: 11px; color: #1B1F24;">Raw + Metadata</div>
      </div>
    </div>
    <div style="font-size: 24px; color: #757575; padding: 0 8px;">&#10132;</div>
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFFFFF; border: 2px dashed #757575; border-radius: 8px; padding: 14px 20px; min-width: 110px;">
        <div style="font-weight: bold; color: #757575;">Silver</div>
        <div style="font-size: 11px; color: #757575;">Lab 05</div>
      </div>
    </div>
    <div style="font-size: 24px; color: #757575; padding: 0 8px;">&#10132;</div>
    <div style="text-align: center; padding: 10px;">
      <div style="background: #FFFFFF; border: 2px dashed #757575; border-radius: 8px; padding: 14px 20px; min-width: 110px;">
        <div style="font-weight: bold; color: #757575;">Gold</div>
        <div style="font-size: 11px; color: #757575;">Lab 06</div>
      </div>
    </div>
  </div>
</div>
""")

Data Flow: Source to Bronze 
 
 
 
 Source 
 JSON in ADLS 
 
 
 ➔ 
 
 
 BRONZE 
 Raw + Metadata 
 
 
 ➔ 
 
 
 Silver 
 Lab 05 
 
 
 ➔ 
 
 
 Gold 
 Lab 06

### Bronze Layer Principles

| Principle | Description |
|-----------|-------------|
| **Raw Data** | Store data exactly as received from the source, no business transformations |
| **Audit Columns** | Add metadata like `_ingestion_time`, `_source_file` for traceability |
| **Append-Only** | Never update or delete; each ingestion adds new records |
| **No Transformations** | Cleaning, validation, and enrichment happen in Silver |
| **Schema Preservation** | Keep the original schema; handle evolution with `mergeSchema` |

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">Key Takeaway:</strong> <span style="color: #1B1F24;">The Bronze layer serves as your data lake's "single source of truth" for raw data. By preserving the original format and adding only audit metadata, you maintain full traceability and can always reprocess data if downstream logic changes.</span>
</div>
""")

Key Takeaway: The Bronze layer serves as your data lake's "single source of truth" for raw data. By preserving the original format and adding only audit metadata, you maintain full traceability and can always reprocess data if downstream logic changes.

---

## Section 3: Batch Ingestion

In this section, we will read raw JSON data from ADLS Gen2, add audit columns for data lineage, and write the result as a Delta table in the Bronze layer.

### Exercise 3.1: Explore Raw Data

Before ingesting, let's understand our source data structure.

In [ ]:
# Read the raw JSON data to explore schema and content
sample_df = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .load(SOURCE_PATH)

print("Schema of raw placements data:")
sample_df.printSchema()

print(f"Total records: {sample_df.count()}")
display(sample_df.limit(5))

### Exercise 3.2: Add Audit Columns

Audit columns are essential for data lineage and troubleshooting. They tell us **when** data was ingested and **where** it came from.

**Your task:** Fill in the blanks with the correct PySpark functions.

| Column | Function | Purpose |
|--------|----------|---------|
| `_ingestion_time` | `current_timestamp()` | When the record was ingested |
| `_source_file` | ``input_file_name()`` | Which file the record came from |
| `_processing_time` | `current_timestamp()` | When the record was processed |

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit, col

# Re-read the data (input_file_name() requires reading from source)
raw_df = spark.read \
    .format("json") \
    .option("multiLine", "true") \
    .load(SOURCE_PATH)

# Audit columns: input_file_name() works for batch reads
# For streaming (Auto Loader), use col("_metadata.file_path") instead

bronze_df = raw_df \
    .withColumn("_ingestion_time", current_timestamp()) \
    .withColumn("_source_file", input_file_name()) \
    .withColumn("_processing_time", current_timestamp())

print("Schema with audit columns:")
bronze_df.printSchema()
display(bronze_df.select("placement_id", "_ingestion_time", "_source_file").limit(3))

### Exercise 3.3: Write to Bronze Delta Table

Now we write the enriched DataFrame to Delta format. This becomes our Bronze table.

**Your task:** Fill in the blanks for the write format and mode.

| Parameter | Value | Why |
|-----------|-------| --- |
| `format` | `"delta"` | Delta Lake provides ACID transactions, versioning, and time travel |
| `mode` | `"overwrite"` | First load replaces any existing data (subsequent loads use append) |

In [ ]:
# TODO: Fill in the blanks for format and mode
# HINT: format = "delta", mode = "overwrite" for initial load

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(BRONZE_PATH)

# Register in metastore
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.placements
    USING DELTA LOCATION '{BRONZE_PATH}'
""")

# Verify
result_df = spark.table(f"{DATABASE_NAME}.placements")
print(f"Bronze table records: {result_df.count()}")
display(result_df.limit(5))

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">Key Takeaway:</strong> <span style="color: #1B1F24;">Batch ingestion is ideal for initial data loads or periodic full refreshes. By using <code>overwrite</code> mode for the first load and <code>append</code> for subsequent loads, you control how data accumulates in the Bronze layer.</span>
</div>
""")

Key Takeaway: Batch ingestion is ideal for initial data loads or periodic full refreshes. By using overwrite mode for the first load and append for subsequent loads, you control how data accumulates in the Bronze layer.

---

## Section 4: Auto Loader for Incremental Ingestion

> *"In production, new files arrive continuously. Auto Loader detects new files automatically, processes them exactly once, and handles schema changes -- all without you writing complex file-tracking logic."*

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Auto Loader: How It Works</h3>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 30px; flex-wrap: wrap; margin-top: 15px;">
    <!-- Left: Flow -->
    <div style="text-align: center;">
      <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 15px;">
        <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 12px 18px;">
          <div style="font-weight: bold; color: #1B1F24;">New Files</div>
          <div style="font-size: 11px; color: #757575;">ADLS Gen2</div>
        </div>
        <div style="font-size: 24px; color: #FF3621;">&#10132;</div>
        <div style="background: #1B3A4B; border-radius: 8px; padding: 12px 18px; color: white;">
          <div style="font-weight: bold; color: #FFFFFF;">Auto Loader</div>
          <div style="font-size: 11px; color: #F9F7F4;">cloudFiles</div>
        </div>
        <div style="font-size: 24px; color: #FF3621;">&#10132;</div>
        <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 12px 18px;">
          <div style="font-weight: bold; color: #1B1F24;">Delta Table</div>
          <div style="font-size: 11px; color: #757575;">Bronze Layer</div>
        </div>
      </div>
      <!-- Checkpoint below Auto Loader -->
      <div style="display: flex; justify-content: center;">
        <div style="background: #FFFFFF; border: 2px solid #1B3A4B; border-radius: 8px; padding: 10px 18px;">
          <div style="font-weight: bold; color: #1B3A4B;">Checkpoint</div>
          <div style="font-size: 11px; color: #757575;">Tracks processed files</div>
        </div>
      </div>
    </div>
    <!-- Right: Features -->
    <div style="background: #FFFFFF; border-radius: 8px; padding: 15px 20px; min-width: 220px; border: 1px solid #E0E0E0;">
      <div style="font-weight: bold; color: #1B3A4B; margin-bottom: 8px;">Key Features</div>
      <div style="font-size: 13px; color: #1B1F24; line-height: 1.8;">
        &#10003; Automatic file discovery<br/>
        &#10003; Schema inference and evolution<br/>
        &#10003; Exactly-once processing<br/>
        &#10003; Handles millions of files<br/>
        &#10003; Fault-tolerant with checkpoints
      </div>
    </div>
  </div>
</div>
""")

Auto Loader: How It Works 
 
 <!-- Left: Flow -->
 
 
 
 New Files 
 ADLS Gen2 
 
 ➔ 
 
 Auto Loader 
 cloudFiles 
 
 ➔ 
 
 Delta Table 
 Bronze Layer 
 
 
 <!-- Checkpoint below Auto Loader -->
 
 
 Checkpoint 
 Tracks processed files 
 
 
 
 <!-- Right: Features -->
 
 Key Features 
 
 ✓ Automatic file discovery 
 ✓ Schema inference and evolution 
 ✓ Exactly-once processing 
 ✓ Handles millions of files 
 ✓ Fault-tolerant with checkpoints

### Exercise 4.1: Configure Auto Loader

Auto Loader uses the `cloudFiles` format to set up a streaming source that automatically detects new files.

**Your task:** Fill in the blank with the correct format name.

| Option | Value | Purpose |
|--------|-------|---------|
| `format` | `"cloudFiles"` | Tells Spark to use Auto Loader |
| `cloudFiles.format` | `"json"` | Format of the source files |
| `cloudFiles.schemaLocation` | checkpoint path | Where to store inferred schema |
| `cloudFiles.inferColumnTypes` | `"true"` | Infer proper data types |
| `cloudFiles.schemaEvolutionMode` | `"addNewColumns"` | Handle new columns automatically |

In [ ]:
# TODO: Fill in the blank with the Auto Loader format
# HINT: The format name is "cloudFiles"

auto_loader_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .load(SOURCE_INCREMENTAL)

print("Auto Loader stream configured")

### Exercise 4.2: Add Audit Columns to Stream

Just like batch ingestion, streaming data also needs audit columns for traceability.

In [ ]:
# Add audit columns to the streaming DataFrame
# Note: For streaming, use col("_metadata.file_path") instead of input_file_name()
bronze_stream_df = auto_loader_df \
    .withColumn("_ingestion_time", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .withColumn("_processing_time", current_timestamp())

print("Audit columns added to stream")

### Exercise 4.3: Write Stream to Delta

Now we write the stream to our Bronze Delta table. The `availableNow=True` trigger processes all available files and then stops -- ideal for lab environments.

**Your task:** Fill in the blank for the output format.

| Parameter | Value | Why |
|-----------|-------| --- |
| `format` | `"delta"` | Write to Delta Lake |
| `outputMode` | `"append"` | Add new records without overwriting |
| `checkpointLocation` | checkpoint path | Track which files have been processed |
| `mergeSchema` | `"true"` | Accept new columns if schema evolves |
| `trigger` | `availableNow=True` | Process all available, then stop |

In [ ]:
# TODO: Fill in the blank with the correct write format
# HINT: We are writing to a Delta table

query = bronze_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(BRONZE_PATH)

query.awaitTermination()
print("Auto Loader ingestion completed!")

### Exercise 4.4: Verify Incremental Ingestion

Let's verify that the Auto Loader processed the incremental files and appended them to our Bronze table.

In [ ]:
# Verify incremental ingestion results
result_df = spark.table(f"{DATABASE_NAME}.placements")
print(f"Total records after Auto Loader: {result_df.count()}")

# Show the most recently ingested records
display(result_df.orderBy(col("_ingestion_time").desc()).limit(10))

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">Key Takeaway:</strong> <span style="color: #1B1F24;">Auto Loader (<code>cloudFiles</code>) is the recommended way to ingest files incrementally in Databricks. It automatically tracks which files have been processed via checkpoints, guarantees exactly-once semantics, and handles schema evolution -- eliminating the need for custom file-tracking logic.</span>
</div>
""")

Key Takeaway: Auto Loader ( cloudFiles ) is the recommended way to ingest files incrementally in Databricks. It automatically tracks which files have been processed via checkpoints, guarantees exactly-once semantics, and handles schema evolution -- eliminating the need for custom file-tracking logic.

---

## Section 5: Verify Bronze Layer

Let's inspect the Bronze table to confirm everything is correct before moving on to the Silver layer.

In [ ]:
# View Delta table history -- shows all operations performed on the table
print("Delta Table History:")
display(spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_PATH}`"))

In [ ]:
# Check data distribution by asset class
print("Records by Asset Class:")
display(
    spark.table(f"{DATABASE_NAME}.placements")
    .groupBy("asset_class")
    .count()
    .orderBy("count", ascending=False)
)

In [ ]:
# Check data distribution by entity
print("Records by Entity:")
display(
    spark.table(f"{DATABASE_NAME}.placements")
    .groupBy("axa_entity_id")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">Key Takeaway:</strong> <span style="color: #1B1F24;">Always verify your Bronze layer after ingestion. Checking Delta history confirms what operations were performed, while data distribution analysis helps catch obvious issues early -- before they propagate to Silver and Gold layers.</span>
</div>
""")

Key Takeaway: Always verify your Bronze layer after ingestion. Checking Delta history confirms what operations were performed, while data distribution analysis helps catch obvious issues early -- before they propagate to Silver and Gold layers.

---

## What You Learned

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Lab 04 Summary</h3>
  <table style="width: 100%; border-collapse: collapse; background: #FFFFFF; border-radius: 8px; overflow: hidden;">
    <thead>
      <tr style="background: #1B3A4B; color: #FFFFFF;">
        <th style="padding: 12px 15px; text-align: left;">Concept</th>
        <th style="padding: 12px 15px; text-align: left;">What You Learned</th>
        <th style="padding: 12px 15px; text-align: left;">Key Command / API</th>
      </tr>
    </thead>
    <tbody>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Bronze Layer</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Raw data ingestion with audit metadata</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>.write.format("delta")</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Audit Columns</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Track data lineage and source provenance</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>current_timestamp()</code>, <code>col("_metadata.file_path")</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Auto Loader</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Incremental file processing with automatic detection</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>.format("cloudFiles")</code></td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #CD7F32;">Checkpointing</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Exactly-once processing and fault tolerance</td>
        <td style="padding: 10px 15px; color: #1B1F24;"><code>checkpointLocation</code> option</td>
      </tr>

    </tbody>
  </table>
</div>
""")

Concept,What You Learned,Key Command / API
Bronze Layer,Raw data ingestion with audit metadata,".write.format(""delta"")"
Audit Columns,Track data lineage and source provenance,"current_timestamp(), input_file_name()"
Auto Loader,Incremental file processing with automatic detection,".format(""cloudFiles"")"
Checkpointing,Exactly-once processing and fault tolerance,checkpointLocation option


### Next Steps

After the **Unity Catalog governance** section in class, proceed to **Lab 05 - Silver Layer** where you will:
- Read from the Bronze table you just created
- Apply data quality checks and validation rules
- Clean, transform, and enrich the data
- Write to the Silver layer with proper schema enforcement